In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

C:\Users\user\AppData\Local\Temp\ipykernel_10340\3777615979.py:1: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython display
  from IPython.core.display import display, HTML


# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


In [5]:
data.head()

,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0


### Let's divide the training and test set into two partitions

In [49]:
# Your code
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(data['text'], data['label'], test_size=0.2, random_state=42)

## Data Preprocessing

In [50]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [51]:
import re
import pandas as pd

def clean_html(text):
    # Remove inline JavaScript and CSS tags.
    text = re.sub(r"<script\b[^>]*>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<style\b[^>]*>.*?</style>",   " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Remove HTML comments.
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove all remaining tags.
    text = re.sub(r"<[^>]+>", " ", text)

    return text


X_train = X_train.apply(clean_html)
X_test = X_test.apply(clean_html)

In [52]:
X_train

29     ----------- REGARDS, MR NELSON SMITH.KINDLY RE...
535    I have not been able to reach oscar this am. W...
695    ; Huma Abedin B6I'm checking with Pat on the 5...
557    I can have it announced here on Monday - can't...
836        BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...
                             ...                        
106    7653 2612ADAMA IBRAHIM________________________...
270               What does that mean for our schedules?
860    Dear Friend,My Compliment to you,I guess this ...
435    Dear PRESIDENT=2FDIRECTOR=2C My name is Mr=2E ...
102    Let me know if today or tomorrow works for you...
Name: text, Length: 800, dtype: object

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
        
    # Convert to Lowercase    
    text = text.lower()

    # Remove prefixed 'b'
    text = text.replace("b'", "").replace('b"', "")

    # Remove all the special characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Remove single characters
    text = re.sub(r"\b[a-z]\b", " ", text)

    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text).strip()

    return text

X_train = X_train.apply(clean_text)
X_test = X_test.apply(clean_text)

In [55]:
X_train.head(10)

29     regards mr nelson smith kindly reply me on my ...
535    have not been able to reach oscar this am we a...
695    huma abedin checking with pat on the will work...
557       can have it announced here on monday can today
836    bank of africaagence san pedro bp san pedro co...
596    dear friend my proposal to you will be very su...
165    soneri bank limitedsilver jubilee center cbrit...
918            easy joy powerful yahoo bookmarks toolbar
495    from amr ecollince addoattn ai need your urgen...
824    dear nancy very much want to meet and we ve be...
Name: text, dtype: object

## Now let's work on removing stopwords
Remove the stopwords.

In [ ]:
def remove_stopwords(text):
    stop_words = stopwords.words("english")
    text_split = text.split()

    # Remove stop words.
    new_text = [word for word in text_split if word not in stop_words]

    return " ".join(new_text)

X_train = X_train.apply(remove_stopwords)
X_test = X_test.apply(remove_stopwords)

In [61]:
X_train.head()

29     regards mr nelson smith kindly reply private e...
535           able reach oscar supposed send pdb receive
695    huma abedin checking pat work jack jake rest a...
557                               announced monday today
836    bank africaagence san pedro bp san pedro cote ...
Name: text, dtype: object

## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [68]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

def text_lemmatizer(text):
    # Break sentences into words
    words = word_tokenize(text)

    # Use lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmas = [lemmatizer.lemmatize(word, "v") for word in words]
    
    return " ".join(lemmas)

X_train = X_train.apply(text_lemmatizer)

## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Initialize the vectorizer
vectorizer = CountVectorizer()

# Count BoW matrix with CountVectorizer
X_train_bow = vectorizer.fit_transform(X_train)
X_test_bow = vectorizer.transform(X_test)

# Create a bow dataframe
X_train_bow_df = pd.DataFrame(X_train_bow.toarray(),columns=vectorizer.get_feature_names_out())
X_train_bow_df['label'] = y_train.reset_index(drop=True)

In [79]:
def top_words(df, value):
    return (df[df["label"] == value]  
            .drop(columns="label")              
            .sum()                              
            .sort_values(ascending=False)
            .head(10))

print("Top 10 ham words:\n",  top_words(X_train_bow_df, 0.0), "\n")
print("Top 10 spam words:\n", top_words(X_train_bow_df, 1.0))

Top 10 ham words:
 state        117
say          104
pm            97
would         94
call          94
work          91
president     88
mr            84
time          83
percent       80
dtype: int64 

Top 10 spam words:
 money          847
account        760
bank           709
fund           631
us             600
transfer       481
transaction    461
business       424
country        393
contact        389
dtype: int64


## Extra features

In [82]:
# Create new dataframes
data_train = pd.DataFrame({
    "preprocessed_text": X_train.reset_index(drop=True),
    "label": y_train.reset_index(drop=True)
})

data_val = pd.DataFrame({
    "preprocessed_text": X_test.reset_index(drop=True),
    "label": y_test.reset_index(drop=True)
})

In [83]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
0,regard mr nelson smith kindly reply private em...,1,0,0,79
1,able reach oscar suppose send pdb receive,0,0,0,41
2,huma abedin check pat work jack jake rest also...,0,0,0,73
3,announce monday today,0,0,0,21
4,bank africaagence san pedro bp san pedro cote ...,1,1,1,1023


## How would work the Bag of Words with Count Vectorizer concept?

In [84]:
v = CountVectorizer()

X_train_bow = v.fit_transform(X_train)
X_test_bow = v.transform(X_test)

print("X_train", X_train_bow.shape)
print("y_train:", X_test_bow.shape)

X_train (800, 27413)
y_train: (200, 27413)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [85]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the vectorizer
tf_vectorizer = TfidfVectorizer()

X_train_tf = tf_vectorizer.fit_transform(X_train)
X_test_tf = tf_vectorizer.transform(X_test)

print("X_train", X_train_tf.shape)
print("y_train:", X_test_tf.shape)

X_train (800, 27413)
y_train: (200, 27413)


In [87]:
display(X_train_tf)

<800x27413 sparse matrix of type '<class 'numpy.float64'>'
	with 78585 stored elements in Compressed Sparse Row format>

## And the Train a Classifier?

In [89]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()

# learn word probabilities per class
model.fit(X_train_tf, y_train)           
y_pred = model.predict(X_test_tf)    

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      0.88      0.94       125
           1       0.83      1.00      0.91        75

    accuracy                           0.93       200
   macro avg       0.92      0.94      0.92       200
weighted avg       0.94      0.93      0.93       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code